# Optimisations et manipulations

## Note préliminaire sur les données

Les données étant volumineuses (1,5 Go une fois chargées dans une feuille de calcul), nous allons définir une ou plusieurs fonctions par exercice pour créer nos diagrammes.

De cette manière, à la sortie de chacune des fonctions, les variables temporaires utilisées seront supprimées et la mémoire pourra être réutilisée.

## Imports

In [ ]:
!pip install python-geohash

import folium
import geohash
import geopy.distance
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn as sns

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, entre 2014 et 2022.

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!git clone https://github.com/mlambda/dataset-dvf.git

In [ ]:
df = pandas.read_parquet("dataset-dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

In [ ]:
# Pour une raison de lisibilité, on garde le 0 devant les départements à un chiffre
replacedict = {x: "0" + x for x in df.code_departement.unique() if len(x) == 1}
df["code_departement"] = df["code_departement"].cat.rename_categories(replacedict)

df.info()

## Vectorization de fonction avec numpy

On veut déterminer un filtre pour savoir si on est dans un rayon autour de la ville de Nantes

In [ ]:
NANTES_COORDS = 47.218371, -1.553621
folium.Map(location=NANTES_COORDS, zoom_start=14)

Afin d'éviter de tester la france entière, on va limiter notre calcul aux mutations ayant lieu en Loire-Atlantique

In [ ]:
df_small = df[df.code_departement == "44"]
df_small.shape

Nous utiliserons cette fonction de la bibliothèque [`geopy`](https://geopy.readthedocs.io/en/stable/#module-geopy.distance) pour effectuer notre test

In [ ]:
def is_near(x, y, radius=1):
  return geopy.distance.geodesic(NANTES_COORDS, (x, y)).km < radius

En travaillant sur le `df_small`, évaluez la différence de l'approche avec un apply, puis avec une vectorisation

### Solution

In [ ]:
radius = 0.5

In [ ]:
is_near_nantes_serie = df_small[["latitude", "longitude"]].apply(
  lambda x: is_near(x["latitude"], x["longitude"], radius), axis=1
)

In [ ]:
vect_funct = numpy.vectorize(lambda x, y: is_near(x, y, radius))
is_near_nantes_vect_funct = vect_funct(df_small["latitude"], df_small["longitude"])

**Pouquoi il ne semble pas y avoir d'amélioration ?**


**Pourtant on a bien vu des améliorations dans le cours...**

In [ ]:
geohash_serie = df_small[["latitude", "longitude"]].apply(
  lambda x: geohash.encode(x["latitude"], x["longitude"]), axis=1
)

In [ ]:
vect_funct = numpy.vectorize(lambda x, y: geohash.encode(x, y))
geohash = vect_funct(df_small["latitude"], df_small["longitude"])

In [ ]:
df_small["geohash"] = geohash
df_small = df_small[is_near_nantes_serie]

Pour qu'il y ait des améliorations possible par la vectorisation de fonction, il faut que la bibliothèque s'y prête !

## Prix des maisons et des appartements

Dans cet exercice, nous allons calculer le prix au mètre carré des appartements et maisons séparément et afficher l'évolution de ces prix dans l'intervalle considéré.

- *Pour les appartements, filtrez la feuille de calcul de travail pour ne conserver que les lignes où :*
    - *`nombre_appartements`, `valeur_fonciere` & `surface_reelle_bati_appartements` sont strictement positifs.*
    - *`nombre_maisons` est nul.*
- *De la même manière, pour les maisons, filtrez la feuille de calcul de travail pour ne conserver que les lignes où :*
    - *`nombre_maisons`, `valeur_fonciere` & `surface_reelle_bati_maisons` sont strictement positifs.*
    - *`nombre_appartements` est nul.*
- *Utilisez ces données filtrées pour calculer le prix au mètre carré pour chaque mois disponible dans le jeu de données.*
- *Affichez le résultat à l'aide de la fonction [seaborn.lineplot](https://seaborn.pydata.org/generated/seaborn.lineplot.html).*

In [ ]:
def houses_apartments_prices() -> None:
  pass  # Votre code ici


houses_apartments_prices()

### Solution

In [ ]:
def houses_apartments_prices() -> None:
  # On calcule le prix moyen, groupé par date, pour les maisons
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )
  house_prices = (
    df[mask_houses]
    .groupby(df["date"].dt.to_period("M"))
    .apply(lambda x: sum(x.valeur_fonciere) / sum(x.surface_reelle_bati_maisons))
    .rename("houses")
  )
  # On calcule le prix moyen, groupé par date, pour les appartements
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  apt_prices = (
    df[mask_apartments]
    .groupby(df["date"].dt.to_period("M"))
    .apply(lambda x: sum(x.valeur_fonciere) / sum(x.surface_reelle_bati_appartements))
    .rename("apartments")
  )
  # On concatene les résultats pour l'affichage dasn seaborn
  data = pandas.concat([house_prices, apt_prices], axis=1)
  # La date devient une colonne, un nouvel index est créé
  data.reset_index(inplace=True)
  # Les deux séries ayant la même unité, on peut reformuler notre dataframe
  # avec une colonne catégorielle à deux modalités : "houses" et "apartments",
  # et une colonne contenant le prix moyen
  data = data.melt(id_vars=["date"], value_vars=["houses", "apartments"])
  # seaborn ne gère pas les objets de type numpy.datetime64
  data.date = data.date.astype("str")

  # Affichage
  fig, ax = plt.subplots()
  sns.lineplot(data, x="date", y="value", hue="variable", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Valeur moyenne")
  ax.set_title("Prix au mètre carré")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(
    matplotlib.ticker.EngFormatter(unit="€/m²", places=1)
  )
  fig.show()


houses_apartments_prices()

## Corrélations entre variables

Nous allons maintenant étudier les corrélations linéaires avec la valeur foncière et plus généralement entre variables.

- *Calculez les corrélations linéaires de toutes les variables avec `valeur_fonciere` à l'aide de la fonction [`pandas.DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html).*
- *Triez les variables dans l'ordre de la plus corrélée (ou anti-corrélée) à la moins corrélée avec `valeur_fonciere`. Vous pourrez pour cela utiliser [`pandas.Series.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.sort_values.html) et son argument `key` couplé à la fonction [`abs`](https://docs.python.org/fr/3/library/functions.html#abs).*
- *Calculez la matrice de corrélation de ces variables à l'aide de la fonction [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html).*
- *Affichez cette matrice avec la fonction [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html).*

In [ ]:
def corrs() -> None:
  pass  # Votre code ici


corrs()

### Solution

In [ ]:
def corrs() -> None:
  corrs_to_price = (
    df.select_dtypes(include=["number"])
    .corrwith(df.valeur_fonciere)
    .sort_values(ascending=False, key=abs)
  )
  corrs = df.loc[:, corrs_to_price.index].corr()

  fig, ax = plt.subplots(figsize=(15, 10))
  sns.heatmap(corrs, vmin=-1, vmax=1, cmap="vlag", ax=ax)
  ax.set_title("Corrélations entre variables triées par corrélation au prix")
  fig.show()


corrs()